# Section Value Analysis in a Library Dataset
## Cleaning and standardizing the `section` column

**Author:** Vladimir Drešević  
**Notebook type:** Data cleaning and exploratory analysis  
**Language:** English  
**Created:** 2026-04-20

---


## Goal of the analysis

The goal of this notebook is to inspect the values stored in the `section` column, identify inconsistent category names, and standardize them so that the dataset uses a clean and consistent set of library sections.

A clean `section` column is important because inconsistent labels can:
- split counts across multiple names that mean the same thing,
- produce misleading summaries and charts,
- make filtering and grouping less reliable.

## Expected outcome

By the end of this notebook, we expect to:
1. review the current section values,
2. identify section labels that should be merged,
3. apply a cleaning rule using a mapping dictionary,
4. convert the cleaned column to the `category` data type,
5. validate the final values against the official list of library sections.


## Official library sections

The library is expected to contain the following sections:

- Fiction
- Humanities
- Non-Fiction
- Children
- Young Adult
- Reference
- History
- Science
- Rare Books


## Step 1 - Loading data

This notebook assumes that a file named `books_columns_renamed.csv` exists in a folder `data`.

Then, you can use this code to load the dataset:


In [30]:
import pandas as pd

df = pd.read_csv("../data/books_columns_renamed.csv")

## Step 2 – Validate the required column

Before starting the analysis, we should confirm that the dataset contains the `section` column.


In [31]:
if "section" not in df.columns:
    raise KeyError("The DataFrame does not contain a 'section' column.")

print("The 'section' column is available.")

The 'section' column is available.


## Step 3 – Analyze the current values in the `section` column

We begin by extracting the unique values from the section column.
This helps us identify which section labels are currently present in the dataset.


In [32]:
unique_sections = df["section"].unique()
unique_sections

<StringArray>
[           'Fiction',            'History',           'Children',
        'Non-Fiction',            'Science',         'Humanities',
   'Young Adult (YA)',          'Reference',         'Children's',
                  nan,        'Young Adult', 'Children's Fiction',
         'Rare Books']
Length: 13, dtype: str

## Step 4 – Interpret the result

After printing the value counts, we should inspect the output carefully.

### What we are looking for
We want to check whether the dataset contains only the official section names or whether some labels appear in alternative forms.

### Conclusion
Dataset contains naming inconsistencies:

- `Young Adult (YA)` instead of `Young Adult`
- `Children's` instead of `Children`
- `Children's Fiction` instead of `Children`

These labels do not represent truly different sections.  
They are simply alternative names for existing categories, which means they should be standardized before any further analysis.

### Why this matters
If we do not clean these values:
- the same section may be counted as multiple categories,
- charts may contain redundant labels,
- conclusions about the distribution of books across sections may become inaccurate.


## Step 5 – Define the cleaning rules

The following mapping dictionary will merge inconsistent names into the official section labels.


In [33]:
mapping = {
    "Young Adult (YA)": "Young Adult",
    "Children's": "Children",
    "Children's Fiction": "Children"
}

## Step 6 – Apply the cleaning rules

Now we apply the mapping and convert the column to the `category` data type.

This gives us:
- standardized labels,
- a cleaner dataset,
- a more memory-efficient representation of repeated categorical values.


In [34]:
df["section"] = df["section"].astype(object).replace(mapping).astype("category")

## Step 7 – Recheck the section counts after cleaning

After standardization, we should review the unique values again to confirm that the categories have been merged correctly.


In [35]:
unique_sections = df["section"].unique()
print(unique_sections)

['Fiction', 'History', 'Children', 'Non-Fiction', 'Science', 'Humanities', 'Young Adult', 'Reference', NaN, 'Rare Books']
Categories (9, str): ['Children', 'Fiction', 'History', 'Humanities', ..., 'Rare Books', 'Reference', 'Science', 'Young Adult']


## Step 8 – Compare the cleaned values with the official list

This step helps us detect:
- missing expected sections,
- unexpected labels that still remain after cleaning.


In [36]:
expected_sections = [
    "Fiction",
    "Humanities",
    "Non-Fiction",
    "Children",
    "Young Adult",
    "Reference",
    "History",
    "Science",
    "Rare Books"
]

current_sections = sorted(df["section"].dropna().astype(str).unique())

missing_sections = sorted(set(expected_sections) - set(current_sections))
unexpected_sections = sorted(set(current_sections) - set(expected_sections))

print("Current sections:")
print(current_sections)

print("\nMissing expected sections:")
print(missing_sections)

print("\nUnexpected sections after cleaning:")
print(unexpected_sections)

Current sections:
['Children', 'Fiction', 'History', 'Humanities', 'Non-Fiction', 'Rare Books', 'Reference', 'Science', 'Young Adult']

Missing expected sections:
[]

Unexpected sections after cleaning:
[]


## Step 9 – Final conclusion

After applying the mapping rules, the `section` column should be more consistent and better aligned with the official library structure.

### Final takeaway
This analysis shows an important principle of data cleaning:

> Before creating summaries, charts, or reports, categorical values should be standardized.

In this example, several labels referred to the same logical section.  
By merging them into a single official name, we improve the quality of future analysis.

### Recommended next steps
After this cleaning step, the dataset is ready for tasks such as:
- section-based summaries,
- bar chart visualizations,
- comparisons between library sections,
- grouping and aggregation by category.
